# V7.2 — FO theo kích thước: chẩn đoán → build → train → val gate → test (RTX 4060)

**Vì sao có V7.2:** V7.1 thêm khoảng 3.000 box `foreign_object` (FO) import có area trung vị 0.025, trong khi FO ở val/test có area trung vị 0.09–0.12 (train base: 0.037). Model bị kéo về FO nhỏ, nên FO AP50-95 giảm cả trên val (−0.053) lẫn test (−0.040).

**V7.2 thêm FO cỡ vừa/lớn và giới hạn số lượng.** Mỗi biến thể fine-tune từ seed85 với hyperparameter y hệt V7.1:

| Biến thể | Train |
|---|---|
| `a_base_only` | chỉ ảnh base. Gần như trùng dữ liệu seed85 (khác 12 label), **mặc định không train** |
| `b_fo_large` | a + ảnh import có FO, mọi FO có area ≥ `MIN_FO_AREA`, không có box cách điện, tối đa `MAX_IMPORT_FO` box |
| `c_fo_large_rep` | b + nhân bản ảnh có FO area ≥ `LARGE_FO_AREA` |

Val/test giữ nguyên. Chọn checkpoint trên val; test chỉ chạy cho bản PASS val gate.

Chạy lần lượt từ trên xuống. Mỗi cell chạy lại được an toàn: dataset đã có thì bỏ qua, train dang dở thì resume, train xong rồi thì bỏ qua.

In [1]:
# Cấu hình chính
from pathlib import Path
import os, subprocess, sys, json, time

EXPERIMENT = 'finding_v7p2_211126'
REMOTE_ROOT = f'stash://bunpmc/projects/{EXPERIMENT}'
SEED85_REMOTE = 'stash://bunpmc/projects/finding_seed/seed85/curated_yolo26m/weights/best.pt'
V7P1_REMOTE = 'stash://bunpmc/projects/finding_v7p1_171126/yolo26m_finetune_v2_reviewed_181126/weights/best.pt'

# a_base_only gần như trùng dữ liệu train của seed85 (chỉ khác 12 label) nên không train.
# Muốn bỏ một bản thì xoá khỏi list.
VARIANTS = ['b_fo_large', 'c_fo_large_rep']
MIN_FO_AREA = 0.03     # FO import: mọi box phải >= ngưỡng này (val/test median ~0.09-0.12)
MAX_IMPORT_FO = 1200   # tối đa số box FO import
LARGE_FO_AREA = 0.05   # biến thể c: ảnh có FO >= ngưỡng này được nhân bản
REPEAT = 1             # số bản sao thêm cho biến thể c
FO_TOLERANCE = 0.0     # val gate: FO AP50-95 được phép giảm tối đa bao nhiêu

EPOCHS, IMGSZ, BATCH, SEED, SYNC_EVERY = 30, 640, 8, 85, 5
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

def find_repo():
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / 'benchmakring_transmission_line_defect_model').is_dir():
            return p
    raise RuntimeError('Mở notebook từ trong repo train')

ROOT = find_repo()
os.chdir(ROOT)
BENCH = ROOT / 'benchmakring_transmission_line_defect_model'
SCRIPTS = BENCH / 'scripts'
DATA_DIR = BENCH / 'data'
SOURCE = DATA_DIR / 'finding_nemo_human_reviewed_6cls_v2_171126'
INVENTORY = ROOT / 'artifacts/dataset_audit/v7p1_171126_imported_annotation_audit/image_inventory.csv'
PREFIX = 'finding_nemo_6cls_v7p2'
PROJECT = ROOT / 'runs' / EXPERIMENT
OUT = BENCH / 'results' / EXPERIMENT
SEED85 = ROOT / 'weights/v7p1_seed85_curated_yolo26m_best.pt'
V7P1 = ROOT / 'runs/finding_v7p1_171126/yolo26m_finetune_v2_reviewed_181126/weights/best.pt'

for path in (SOURCE / 'REVIEW_APPLIED_181126.json', INVENTORY,
             SCRIPTS / 'build_v7p2_fo_datasets.py', SCRIPTS / 'eval_v7p2_fo.py'):
    assert path.is_file(), f'Thiếu {path} (đã git pull nhánh finding_v7p1_171126 chưa?)'
PROJECT.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)

def run_py(script, *args):
    # Chạy script bằng chính Python của kernel và in output trực tiếp vào notebook.
    cmd = [sys.executable, '-u', str(SCRIPTS / script), *map(str, args)]
    print('$', ' '.join(cmd[2:]), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         encoding='utf-8', errors='replace', cwd=ROOT)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait():
        raise RuntimeError(f'{script} lỗi (exit {p.returncode}), xem output phía trên')

def data_yaml(variant):
    return DATA_DIR / f'{PREFIX}_{variant}' / 'data_local.yaml'

print('ROOT   :', ROOT)
print('SOURCE :', SOURCE)
print('OUT    :', OUT)

ROOT   : D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection
SOURCE : D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\data\finding_nemo_human_reviewed_6cls_v2_171126
OUT    : D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\results\finding_v7p2_211126


## 1. Môi trường
Nếu import lỗi, bỏ comment dòng `%pip`, chạy cell rồi restart kernel. Dùng cùng kernel `.venv` với notebook V7.1.

In [2]:
# %pip install -U "ultralytics==8.4.172" python-dotenv pandas
import torch, ultralytics, yaml, pandas as pd
from ultralytics import YOLO
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / '.env')
except ImportError:
    pass
assert torch.cuda.is_available(), 'Không thấy CUDA. Kiểm tra PyTorch CUDA/driver NVIDIA.'
print('Python     :', sys.executable)
print('Ultralytics:', ultralytics.__version__)
print('PyTorch    :', torch.__version__)
print('GPU        :', torch.cuda.get_device_name(0))

Python     : d:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\.venv\Scripts\python.exe
Ultralytics: 8.4.172
PyTorch    : 2.14.1+cu126
GPU        : NVIDIA GeForce RTX 4060 Laptop GPU


## 2. Camber và checkpoint
Tải seed85 và V7.1 best nếu local chưa có. Cần `CAMBER_API_KEY` trong `.env`.

In [3]:
def load_camber_key():
    if os.environ.get('CAMBER_API_KEY'):
        return
    env_file = ROOT / '.env'
    if env_file.is_file():
        for line in env_file.read_text(encoding='utf-8').splitlines():
            key, sep, value = line.partition('=')
            if sep and key.strip() == 'CAMBER_API_KEY' and value.strip():
                os.environ['CAMBER_API_KEY'] = value.strip().strip(chr(34)).strip(chr(39)); return
    raise RuntimeError('Thiếu CAMBER_API_KEY trong environment hoặc .env của repo')

def wsl_path(path: Path):
    path = path.resolve()
    return f'/mnt/{path.drive.rstrip(chr(58)).lower()}{path.as_posix().split(chr(58),1)[1]}'

CAMBER = '/home/dotru/.camber/bin/camber'
load_camber_key()
CAMBER_ENV = os.environ.copy()
wslenv = [x for x in CAMBER_ENV.get('WSLENV', '').split(':') if x]
if 'CAMBER_API_KEY' not in wslenv: wslenv.append('CAMBER_API_KEY')
CAMBER_ENV['WSLENV'] = ':'.join(wslenv)

def camber(*parts, check=True):
    converted = [wsl_path(x) if isinstance(x, Path) else str(x) for x in parts]
    p = subprocess.run(['wsl', '--', CAMBER, 'stash', *converted], env=CAMBER_ENV,
                       capture_output=True, text=True, encoding='utf-8', errors='replace')
    if check and p.returncode:
        raise RuntimeError((p.stdout or p.stderr)[-3000:])
    return p

def ensure_remote_dir(uri):
    if camber('ls', uri, check=False).returncode != 0:
        made = camber('mkdir', uri, check=False)
        if made.returncode != 0 and camber('ls', uri, check=False).returncode != 0:
            raise RuntimeError((made.stdout or made.stderr)[-1200:])

def sync_file(local: Path, remote: str):
    if local.is_file():
        camber('cp', local, remote)

auth = subprocess.run(['wsl', '--', CAMBER, 'me'], env=CAMBER_ENV, capture_output=True,
                      text=True, encoding='utf-8', errors='replace')
assert auth.returncode == 0, (auth.stdout or auth.stderr)[-1200:]
print('Camber OK')
ensure_remote_dir('stash://bunpmc/projects'); ensure_remote_dir(REMOTE_ROOT)

for local, remote in ((SEED85, SEED85_REMOTE), (V7P1, V7P1_REMOTE)):
    if not local.is_file():
        local.parent.mkdir(parents=True, exist_ok=True)
        print('Downloading', remote); camber('cp', remote, local)
    assert local.stat().st_size > 40_000_000, local
    print(local.relative_to(ROOT), round(local.stat().st_size / 2**20, 1), 'MiB')

Camber OK
weights\v7p1_seed85_curated_yolo26m_best.pt 42.0 MiB
runs\finding_v7p1_171126\yolo26m_finetune_v2_reviewed_181126\weights\best.pt 42.0 MiB


## 3. Chẩn đoán FO → insulator_broken trên val (không train)
So seed85 với V7.1 trên val. Cần xem:
- `fo_by_area`: FO bị nhận nhầm hoặc bỏ sót ở nhóm kích thước nào.
- `confusion_*`: hàng `foreign_object`, cột `insulator_broken`.
- `gallery/`: ảnh cụ thể. Xanh là GT, đỏ là dự đoán.

In [4]:
DIAG = OUT / 'diag_val'
if not (DIAG / 'eval_results.json').is_file():
    run_py('eval_v7p2_fo.py', '--data', SOURCE / 'data_local.yaml',
           '--model', f'seed85={SEED85}', '--model', f'v7p1={V7P1}', '--out', DIAG)
display(pd.read_csv(DIAG / 'fo_by_area.csv'))
for name in ('seed85', 'v7p1'):
    print(f'\nConfusion {name} / val (hàng = GT, cột = dự đoán)')
    display(pd.read_csv(DIAG / f'confusion_{name}_val.csv', index_col=0))
print('Gallery:', DIAG / 'gallery')

,model,split,area_bin,fo_gt,recall_at_conf,as_insulator_broken,as_other_class,missed
0,seed85,val,small_lt1pct,10,0.9000,0,0,1
1,seed85,val,medium_1to5pct,12,0.5833,0,0,5
2,seed85,val,large_ge5pct,40,0.8250,0,0,7
3,seed85,val,all,62,0.7903,0,0,13
4,v7p1,val,small_lt1pct,10,0.7000,0,0,3
5,v7p1,val,medium_1to5pct,12,0.5000,0,0,6
6,v7p1,val,large_ge5pct,40,0.8250,0,0,7
7,v7p1,val,all,62,0.7419,0,0,16



Confusion seed85 / val (hàng = GT, cột = dự đoán)


,insulator,insulator_broken,insulator_flashover,bird_nest,foreign_object,broken_strand,missed
gt\pred,,,,,,,
insulator,981,0,0,0,0,0,106.0
insulator_broken,0,650,0,0,0,0,160.0
insulator_flashover,0,1,127,0,0,0,35.0
bird_nest,0,0,0,755,0,0,75.0
foreign_object,0,0,0,0,49,0,13.0
broken_strand,0,0,0,0,0,182,70.0
background_fp,171,172,40,94,20,44,NaN



Confusion v7p1 / val (hàng = GT, cột = dự đoán)


,insulator,insulator_broken,insulator_flashover,bird_nest,foreign_object,broken_strand,missed
gt\pred,,,,,,,
insulator,980,0,0,0,0,0,107.0
insulator_broken,0,640,0,0,0,0,170.0
insulator_flashover,0,1,123,0,0,0,39.0
bird_nest,0,0,0,754,0,0,76.0
foreign_object,0,0,0,0,46,0,16.0
broken_strand,0,0,0,0,1,182,69.0
background_fp,179,160,37,91,24,45,NaN


Gallery: D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\results\finding_v7p2_211126\diag_val\gallery


## 4. Build dataset V7.2
Đọc dòng `Import selection`. Nếu số FO box import được chọn quá ít (< ~300), hạ `MIN_FO_AREA` xuống 0.02, xoá các thư mục `finding_nemo_6cls_v7p2_*` rồi chạy lại cell này.

In [5]:
missing = [v for v in VARIANTS if not data_yaml(v).is_file()]
if missing:
    run_py('build_v7p2_fo_datasets.py', '--source', SOURCE, '--inventory', INVENTORY,
           '--output-root', DATA_DIR, '--prefix', PREFIX, '--variants', *missing,
           '--min-fo-area', MIN_FO_AREA, '--max-import-fo', MAX_IMPORT_FO,
           '--large-fo-area', LARGE_FO_AREA, '--repeat', REPEAT, '--seed', SEED)
rows = []
for v in VARIANTS:
    b = json.loads((DATA_DIR / f'{PREFIX}_{v}' / 'BUILD_V7P2.json').read_text(encoding='utf-8'))
    assert not b['validation_problems'], (v, b['validation_problems'][:5])
    tr = b['stats']['train']
    rows.append({'variant': v, 'train_images': tr['images'], **{f'FO_{k}': n for k, n in tr['fo_area_bins'].items()},
                 'FO_total': tr['boxes']['foreign_object'], 'IB': tr['boxes']['insulator_broken'],
                 **{f'add_{k}': n for k, n in b['train_additions'].items()}})
print('Import selection:', b['import_selection'])
display(pd.DataFrame(rows))

Import selection: {'has_small_fo': 1002, 'no_fo': 1084, 'has_insulator_or_other_class': 18, 'selected': 960}


,variant,train_images,FO_small_lt1pct,FO_medium_1to5pct,FO_large_ge5pct,FO_total,IB,add_imported_fo_large,add_oversample_large_fo
0,b_fo_large,18845,204,630,1201,2035,6412,960,NaN
1,c_fo_large_rep,19993,204,691,2402,3297,6412,960,1148.0


## 5. Train từng biến thể
Mỗi biến thể là một run riêng trong `runs/finding_v7p2_211126/`. Có `last.pt` thì resume; có `DONE.json` thì bỏ qua. Cứ 5 epoch đồng bộ lên Camber.

Mỗi biến thể mất khoảng như một lần train V7.1; mặc định train b rồi c. Nếu chỉ có thời gian cho một bản, để `VARIANTS = ['b_fo_large']`.

In [6]:
import gc

def sync_run(run_dir, remote, epoch=None):
    ensure_remote_dir(remote); ensure_remote_dir(remote + '/weights')
    for rel in ('results.csv', 'args.yaml', 'DONE.json', 'weights/last.pt', 'weights/best.pt'):
        sync_file(run_dir / rel, f'{remote}/{rel}')
    if epoch:
        sync_file(run_dir / f'weights/epoch{epoch}.pt', f'{remote}/weights/epoch{epoch}.pt')
    print(f'[CAMBER SYNC] epoch={epoch} -> {remote}', flush=True)

def try_restore(remote, local):
    if local.exists():
        return
    local.parent.mkdir(parents=True, exist_ok=True)
    if camber('cp', remote, local, check=False).returncode == 0 and local.exists():
        print('[RESTORED]', local)

def train_variant(variant):
    name = f'yolo26m_v7p2_{variant}'
    run_dir, remote = PROJECT / name, f'{REMOTE_ROOT}/{name}'
    if (run_dir / 'DONE.json').is_file():
        print(f'[SKIP] {variant} đã train xong'); return
    last = run_dir / 'weights/last.pt'
    for rel in ('weights/last.pt', 'results.csv', 'args.yaml'):
        try_restore(f'{remote}/{rel}', run_dir / rel)
    gc.collect(); torch.cuda.empty_cache()
    model = YOLO(str(last if last.is_file() else SEED85))
    clock = time.time()

    def on_fit_epoch_end(trainer):
        m = trainer.metrics or {}
        f = lambda k: f"{float(m.get(k, float('nan'))):.4f}"
        print(f'\n[V7.2 {variant}] epoch {trainer.epoch + 1}/{trainer.epochs} | '
              f'val mAP50={f("metrics/mAP50(B)")} | mAP50-95={f("metrics/mAP50-95(B)")} | '
              f'{(time.time() - clock) / 60:.1f} min', flush=True)

    def on_model_save(trainer):
        e = trainer.epoch + 1
        if e % SYNC_EVERY == 0 or e == trainer.epochs:
            sync_run(run_dir, remote, epoch=e)

    model.add_callback('on_fit_epoch_end', on_fit_epoch_end)
    model.add_callback('on_model_save', on_model_save)
    if last.is_file():
        print('RESUME:', last)
        model.train(resume=True, data=str(data_yaml(variant).resolve()))
    else:
        print(f'START {variant} từ seed85, data = {data_yaml(variant)}')
        model.train(
            data=str(data_yaml(variant)), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0,
            optimizer='AdamW', lr0=1e-4, lrf=0.01, cos_lr=True, warmup_epochs=1.0,
            momentum=0.937, weight_decay=5e-4,
            mosaic=1.0, close_mosaic=10, fliplr=0.5, flipud=0.0, translate=0.1, scale=0.5,
            degrees=0.0, shear=0.0, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, erasing=0.4,
            amp=True, seed=SEED, deterministic=True, workers=2, cache=False,
            project=str(PROJECT), name=name, exist_ok=True,
            save=True, save_period=SYNC_EVERY, plots=True, patience=12, val=True, verbose=True,
        )
    assert (run_dir / 'weights/best.pt').is_file(), run_dir
    (run_dir / 'DONE.json').write_text(json.dumps({
        'variant': variant, 'data': str(data_yaml(variant)), 'init': str(SEED85),
        'finished_at': time.strftime('%Y-%m-%d %H:%M:%S')}, indent=2), encoding='utf-8')
    sync_run(run_dir, remote)
    del model; gc.collect(); torch.cuda.empty_cache()

for v in VARIANTS:
    train_variant(v)
print('Đã train xong:', [v for v in VARIANTS if (PROJECT / f'yolo26m_v7p2_{v}' / 'DONE.json').is_file()])

[SKIP] b_fo_large đã train xong
RESUME: D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\runs\finding_v7p2_211126\yolo26m_v7p2_c_fo_large_rep\weights\last.pt
New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.172  Python-3.12.2 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\data\finding_nemo_6cls_v7p2_c_fo_large_rep\data_local.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, d

## 6. Val gate so với seed85
PASS khi val mAP50-95 ≥ seed85 **và** FO AP50-95 không giảm quá `FO_TOLERANCE`. Có cả V7.1 để so. Test chưa được đụng ở bước này.

In [7]:
trained = [v for v in VARIANTS if (PROJECT / f'yolo26m_v7p2_{v}' / 'DONE.json').is_file()]
assert trained, 'Chưa có biến thể nào train xong'
best_pt = {v: PROJECT / f'yolo26m_v7p2_{v}' / 'weights/best.pt' for v in trained}
VAL = OUT / 'eval_val'
models = ['--model', f'seed85={SEED85}', '--model', f'v7p1={V7P1}']
for v in trained:
    models += ['--model', f'{v}={best_pt[v]}']
run_py('eval_v7p2_fo.py', '--data', data_yaml(trained[0]), *models, '--split', 'val',
       '--fo-tolerance', FO_TOLERANCE, '--out', VAL)
gate = pd.read_csv(VAL / 'gate.csv')
display(gate)
display(pd.read_csv(VAL / 'summary.csv'))
display(pd.read_csv(VAL / 'per_class.csv').pivot(index='class', columns='model', values='AP50_95').round(4))
display(pd.read_csv(VAL / 'fo_by_area.csv'))
PASSED = [v for v in gate[gate.gate == 'PASS'].model if v in best_pt]
print('PASS val gate:', PASSED or 'không có — giữ seed85')

$ D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\scripts\eval_v7p2_fo.py --data D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\data\finding_nemo_6cls_v7p2_b_fo_large\data_local.yaml --model seed85=D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\weights\v7p1_seed85_curated_yolo26m_best.pt --model v7p1=D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\runs\finding_v7p1_171126\yolo26m_finetune_v2_reviewed_181126\weights\best.pt --model b_fo_large=D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\runs\finding_v7p2_211126\yolo26m_v7p2_b_fo_large\weights\best.pt --model c_fo_large_rep=D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\runs\finding_v7p2_211126\yolo26m_v7p2_c_fo_large_rep\weights\best.pt --split val --fo-tolerance 0.0 

,model,baseline,d_mAP50,d_mAP50_95,d_FO_AP50_95,classes_improved,gate
0,v7p1,seed85,-0.0010,-0.0089,-0.0531,broken_strand,FAIL
1,b_fo_large,seed85,0.0061,-0.0003,0.0084,foreign_object|broken_strand,FAIL
2,c_fo_large_rep,seed85,0.0023,-0.0008,-0.0044,broken_strand,FAIL


,model,split,precision,recall,mAP50,mAP50_95
0,seed85,val,0.878122,0.770926,0.840565,0.479695
1,v7p1,val,0.860558,0.778105,0.839601,0.470763
2,b_fo_large,val,0.863773,0.790551,0.846690,0.479346
3,c_fo_large_rep,val,0.864114,0.781284,0.842897,0.478865


model,b_fo_large,c_fo_large_rep,seed85,v7p1
class,,,,
bird_nest,0.5676,0.5688,0.5704,0.5691
broken_strand,0.3746,0.3792,0.3716,0.3801
foreign_object,0.4929,0.4802,0.4846,0.4314
insulator,0.6839,0.6857,0.6885,0.6852
insulator_broken,0.4233,0.4242,0.4261,0.4259
insulator_flashover,0.3338,0.3350,0.3370,0.3330


,model,split,area_bin,fo_gt,recall_at_conf,as_insulator_broken,as_other_class,missed
0,seed85,val,small_lt1pct,10,0.9000,0,0,1
1,seed85,val,medium_1to5pct,12,0.5833,0,0,5
2,seed85,val,large_ge5pct,40,0.8250,0,0,7
3,seed85,val,all,62,0.7903,0,0,13
4,v7p1,val,small_lt1pct,10,0.7000,0,0,3
5,v7p1,val,medium_1to5pct,12,0.5000,0,0,6
6,v7p1,val,large_ge5pct,40,0.8250,0,0,7
7,v7p1,val,all,62,0.7419,0,0,16
8,b_fo_large,val,small_lt1pct,10,0.9000,0,0,1
9,b_fo_large,val,medium_1to5pct,12,0.6667,0,0,4


PASS val gate: không có — giữ seed85


## 7. Test cố định (chỉ cho bản PASS)
Chỉ chạy một lần. Không dùng kết quả test để chỉnh tham số rồi chạy lại.

In [8]:
if not PASSED:
    print('Không có biến thể nào PASS val gate. Giữ seed85, không chạy test.')
else:
    TEST = OUT / 'eval_test'
    models = ['--model', f'seed85={SEED85}']
    for v in PASSED:
        models += ['--model', f'{v}={best_pt[v]}']
    run_py('eval_v7p2_fo.py', '--data', data_yaml(PASSED[0]), *models, '--split', 'test', '--out', TEST)
    s = pd.read_csv(TEST / 'summary.csv').set_index('model')
    display(s)
    display((s.drop(index='seed85')[['precision', 'recall', 'mAP50', 'mAP50_95']]
             - s.loc['seed85', ['precision', 'recall', 'mAP50', 'mAP50_95']]).round(4))
    display(pd.read_csv(TEST / 'per_class.csv').pivot(index='class', columns='model', values='AP50_95').round(4))
    display(pd.read_csv(TEST / 'fo_by_area.csv'))

Không có biến thể nào PASS val gate. Giữ seed85, không chạy test.


## 8. Upload kết quả lên Camber

In [9]:
for sub in ('diag_val', 'eval_val', 'eval_test'):
    folder = OUT / sub
    if not folder.is_dir():
        continue
    remote = f'{REMOTE_ROOT}/evaluation/{sub}'
    ensure_remote_dir(f'{REMOTE_ROOT}/evaluation'); ensure_remote_dir(remote)
    for f in sorted(folder.glob('*.csv')) + sorted(folder.glob('*.json')):
        sync_file(f, f'{remote}/{f.name}')
    print('Uploaded', sub, '->', remote)

Uploaded diag_val -> stash://bunpmc/projects/finding_v7p2_211126/evaluation/diag_val
Uploaded eval_val -> stash://bunpmc/projects/finding_v7p2_211126/evaluation/eval_val
